# Auto Insurance Claims: Model Training, Comparison & Selection
This notebook trains both baseline machine learning algorithms (Linear Regression, Decision Tree, Random Forest, Gradient Boosting) and advanced gradient boosting engines (CatBoost, LightGBM, XGBoost) using log-transformed targets and native categorical features, comparing them to save the winner.

## Step 1: Import Libraries and Setup Style

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

sns.set_theme(style="whitegrid")
COLOR_PRIMARY = '#10b981'
COLOR_SECONDARY = '#3b82f6'
COLOR_ACCENT = '#f59e0b'

## Step 2: Load the Cleaned Raw Data and Recreate Split

In [ ]:
df = pd.read_csv('insurance_claims.csv')
df = df.replace('?', 'UNKNOWN')
df = df.fillna('UNKNOWN')
if '_c39' in df.columns:
    df = df.drop(columns=['_c39'])

df['policy_bind_date'] = pd.to_datetime(df['policy_bind_date'])
df['incident_date'] = pd.to_datetime(df['incident_date'])
df['policy_tenure_at_incident'] = (df['incident_date'] - df['policy_bind_date']).dt.days
df['vehicle_age_at_incident'] = df['incident_date'].dt.year - df['auto_year']
df['incident_month'] = df['incident_date'].dt.month
df['incident_day_of_week'] = df['incident_date'].dt.dayofweek

drop_cols = [
    'injury_claim', 'property_claim', 'vehicle_claim', 'fraud_reported',
    'policy_number', 'insured_zip', 'incident_location',
    'policy_bind_date', 'incident_date'
]
target_col = 'total_claim_amount'

X = df.drop(columns=drop_cols + [target_col])
y = df[target_col]

numerical_features = [
    'months_as_customer', 'age', 'policy_deductable', 'policy_annual_premium',
    'umbrella_limit', 'capital-gains', 'capital-loss', 'incident_hour_of_the_day',
    'number_of_vehicles_involved', 'bodily_injuries', 'witnesses', 'auto_year',
    'policy_tenure_at_incident', 'vehicle_age_at_incident', 'incident_month', 'incident_day_of_week'
]
categorical_features = [col for col in X.columns if col not in numerical_features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Recreated data sets. Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## Step 3: Preprocess Datasets for Baseline Models

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

X_train_std = preprocessor.fit_transform(X_train)
X_test_std = preprocessor.transform(X_test)

## Step 4: Preprocess Datasets for Advanced Models (Native Categories & Target Log-Scale)

In [ ]:
# For CatBoost
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()
for col in categorical_features:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

# For LightGBM and XGBoost
X_train_adv = X_train.copy()
X_test_adv = X_test.copy()
for col in categorical_features:
    X_train_adv[col] = X_train_adv[col].astype('category')
    X_test_adv[col] = X_test_adv[col].astype('category')

# Log transform target
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

## Step 5: Fit and Tune Baseline Models

In [ ]:
models_dict = {}

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_std, y_train)
models_dict["Linear Regression"] = lr

# 2. Decision Tree
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=42),
                       {"max_depth": [4, 6, 8, 10], "min_samples_split": [5, 10, 20]},
                       cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
dt_grid.fit(X_train_std, y_train)
models_dict["Decision Tree"] = dt_grid.best_estimator_

# 3. Random Forest
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                       {"n_estimators": [100, 200], "max_depth": [8, 12, None], "min_samples_split": [5, 10]},
                       cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
rf_grid.fit(X_train_std, y_train)
models_dict["Random Forest"] = rf_grid.best_estimator_

# 4. Gradient Boosting
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=42),
                       {"n_estimators": [100, 200], "learning_rate": [0.01, 0.05, 0.1], "max_depth": [3, 4]},
                       cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gb_grid.fit(X_train_std, y_train)
models_dict["Gradient Boosting"] = gb_grid.best_estimator_

print("Baseline models successfully fit.")

## Step 6: Fit Advanced Boosting Models (CatBoost, LightGBM, XGBoost)

In [ ]:
# 5. LightGBM
lgb = LGBMRegressor(n_estimators=300, learning_rate=0.03, max_depth=5, random_state=42, verbose=-1)
lgb.fit(X_train_adv, y_train_log, categorical_feature=categorical_features)
models_dict["LightGBM"] = lgb

# 6. XGBoost
xgb = XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=5, enable_categorical=True, tree_method='hist', random_state=42)
xgb.fit(X_train_adv, y_train_log)
models_dict["XGBoost"] = xgb

# 7. CatBoost
cb = CatBoostRegressor(iterations=600, learning_rate=0.03, depth=6, eval_metric='RMSE', random_seed=42, verbose=0, allow_writing_files=False)
cb.fit(X_train_cb, y_train_log, cat_features=categorical_features, eval_set=(X_test_cb, y_test_log), use_best_model=True)
models_dict["CatBoost"] = cb

print("Advanced boosting models successfully fit.")

## Step 7: Model Evaluation & Benchmarking

In [ ]:
def calculate_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

results = {}
for name, model in models_dict.items():
    # Predict based on model family
    if name in ["Linear Regression", "Decision Tree", "Random Forest", "Gradient Boosting"]:
        y_train_pred = model.predict(X_train_std)
        y_test_pred = model.predict(X_test_std)
    elif name in ["LightGBM", "XGBoost"]:
        y_train_pred = np.expm1(model.predict(X_train_adv))
        y_test_pred = np.expm1(model.predict(X_test_adv))
    else: # CatBoost
        y_train_pred = np.expm1(model.predict(X_train_cb))
        y_test_pred = np.expm1(model.predict(X_test_cb))
        
    # Compute metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_mape = calculate_mape(y_train, y_train_pred)
    test_mape = calculate_mape(y_test, y_test_pred)
    
    results[name] = {
        "Train_RMSE": train_rmse, "Test_RMSE": test_rmse,
        "Train_MAE": train_mae, "Test_MAE": test_mae,
        "Train_R2": train_r2, "Test_R2": test_r2,
        "Train_MAPE": train_mape, "Test_MAPE": test_mape
    }
    
results_df = pd.DataFrame(results).T
results_df

## Step 8: Save the Best Model

In [ ]:
best_name = results_df["Test_RMSE"].idxmin()
best_model = models_dict[best_name]
print(f"Winning Model: {best_name}")

model_path = 'src/processed/absolute_best_model.joblib'
joblib.dump(best_model, model_path)
print(f"Saved best model to {model_path}")

## Step 9: Plot Diagnostics for the Winning Model

In [ ]:
if best_name in ["Linear Regression", "Decision Tree", "Random Forest", "Gradient Boosting"]:
    y_test_pred = best_model.predict(X_test_std)
elif best_name in ["LightGBM", "XGBoost"]:
    y_test_pred = np.expm1(best_model.predict(X_test_adv))
else: # CatBoost
    y_test_pred = np.expm1(best_model.predict(X_test_cb))
    
residuals = y_test - y_test_pred

# Actual vs Predicted Plot
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_test_pred, color=COLOR_SECONDARY, alpha=0.6, edgecolors='w', s=50)
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color=COLOR_ACCENT, linestyle='--', linewidth=2)
plt.title(f"{best_name} - Actual vs Predicted", pad=15)
plt.xlabel("Actual Claim Amount ($)")
plt.ylabel("Predicted Claim Amount ($)")
plt.show()

# Residuals Plot
plt.figure(figsize=(9, 5))
sns.histplot(residuals, kde=True, color=COLOR_PRIMARY, bins=30)
plt.axvline(0, color=COLOR_ACCENT, linestyle='--')
plt.title(f"{best_name} - Residuals Distribution", pad=15)
plt.xlabel("Residual (Actual - Predicted) ($)")
plt.ylabel("Count")
plt.show()